In [1]:
# =============================================================================
# ERASMUS MOBILITY DATA: GEOCODING AND NUTS3 REGIONAL CLASSIFICATION
# =============================================================================
# 
# This notebook performs geocoding of Erasmus mobility data and assigns NUTS3
# regional codes through spatial matching.
#
# INPUT:  Raw Erasmus mobility Excel file (user-specified path)
# OUTPUT: erasmus_with_nuts3.csv - geocoded data with NUTS3 codes
#
# REQUIREMENTS:
# - Raw Erasmus mobility data file
# - NUTS3 shapefile from Eurostat GISCO
#   Download: https://ec.europa.eu/eurostat/cache/GISCO/distribution/v2/nuts/
#   File: NUTS_RG_01M_2021_3035.shp
#
# ESTIMATED RUNTIME: 45-60 minutes (depending on hardware and dataset size)
# =============================================================================

In [ ]:
# =============================================================================
# CELL 1: Import Libraries
# =============================================================================

import pandas as pd
import geopandas as gpd
from geopy.geocoders import Nominatim, ArcGIS
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
import time
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries loaded successfully!")
print(f"  pandas version: {pd.__version__}")
print(f"  geopandas version: {gpd.__version__}")

In [ ]:
# =============================================================================
# CELL 2: Configuration - USER INPUT REQUIRED
# =============================================================================

# ==========================
# CONFIGURE THESE PATHS:
# ==========================

# Path to your Erasmus mobility data file
# Expected format: Excel file with columns 'Sending City', 'Sending Country', 
#                  'Receiving City', 'Receiving Country'
ERASMUS_DATA_PATH = '/path/to/your/Erasmus-KA2-Mobility-Data.xlsx'

# Path to NUTS3 shapefile
# Download from: https://ec.europa.eu/eurostat/cache/GISCO/distribution/v2/nuts/shp/
# Extract and point to the .shp file
NUTS3_SHAPEFILE_PATH = '/path/to/NUTS_RG_01M_2021_3035.shp'

# Output directory (files will be saved here)
OUTPUT_DIR = '/path/to/output/directory/'

# ==========================
# GEOCODING CONFIGURATION:
# ==========================

# Number of parallel threads for geocoding
# Recommended: 20 for M3 MacBook Air, 10-15 for older hardware
MAX_WORKERS = 20

# Nominatim rate limit delay (seconds)
# Required for Nominatim fallback geocoding
NOMINATIM_DELAY = 1

print("=" * 70)
print("CONFIGURATION")
print("=" * 70)
print(f"Erasmus data:  {ERASMUS_DATA_PATH}")
print(f"NUTS3 shapefile: {NUTS3_SHAPEFILE_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Parallel workers: {MAX_WORKERS}")
print("=" * 70)

In [ ]:
# =============================================================================
# CELL 3: Load Erasmus Mobility Data
# =============================================================================

print("=" * 70)
print("LOADING ERASMUS MOBILITY DATA")
print("=" * 70)

# Load the Excel file
df_erasmus = pd.read_excel(ERASMUS_DATA_PATH, sheet_name='full')

print(f"✓ Dataset loaded: {len(df_erasmus):,} records")
print(f"\nColumns in dataset: {df_erasmus.columns.tolist()}")
print(f"\nFirst few rows:")
display(df_erasmus.head())

# Identify unique locations for geocoding
sending_locations = df_erasmus[['Sending City', 'Sending Country']].drop_duplicates()
receiving_locations = df_erasmus[['Receiving City', 'Receiving Country']].drop_duplicates()

print(f"\n✓ Unique sending locations: {len(sending_locations):,}")
print(f"✓ Unique receiving locations: {len(receiving_locations):,}")
print(f"\nTotal unique locations to geocode: {len(sending_locations) + len(receiving_locations):,}")

In [ ]:
# =============================================================================
# CELL 4: Define Geocoding Functions
# =============================================================================

# Initialize geocoders
geolocator_arcgis = ArcGIS(timeout=10)
geolocator_nominatim = Nominatim(user_agent="erasmus_nuts3_geocoder", timeout=10)

def geocode_parallel(city, country_code):
    """
    Geocode city-country pairs using dual-provider strategy.
    
    Strategy:
    1. Try ArcGIS first (no rate limit, fast)
    2. Fall back to Nominatim if ArcGIS fails (with rate limiting)
    
    Parameters:
    -----------
    city : str
        City name
    country_code : str
        Country code or name (e.g., "AT - Austria")
    
    Returns:
    --------
    tuple : (latitude, longitude) or (None, None) if geocoding fails
    """
    if pd.isna(city) or str(city).strip() == "":
        return None, None
    
    # Extract country code (e.g., "AT - Austria" -> "AT")
    if pd.notna(country_code) and " - " in str(country_code):
        country = str(country_code).split(" - ")[0].strip()
    else:
        country = str(country_code).strip()
    
    query = f"{city}, {country}"
    
    # Try ArcGIS first (no rate limit)
    try:
        location = geolocator_arcgis.geocode(query, timeout=8)
        if location:
            return location.latitude, location.longitude
    except Exception:
        pass
    
    # Fallback to Nominatim if ArcGIS fails
    try:
        location = geolocator_nominatim.geocode(query, timeout=8)
        if location:
            time.sleep(NOMINATIM_DELAY)  # Rate limit for Nominatim
            return location.latitude, location.longitude
    except Exception:
        pass
    
    return None, None

print("✓ Geocoding functions defined")
print(f"  Strategy: ArcGIS (primary) → Nominatim (fallback)")
print(f"  Nominatim rate limit: {NOMINATIM_DELAY}s delay")

In [ ]:
# =============================================================================
# CELL 5: Geocode Sending Locations
# =============================================================================

print("=" * 70)
print("GEOCODING SENDING LOCATIONS")
print("=" * 70)
print(f"Total unique sending locations: {len(sending_locations):,}")
print(f"Using {MAX_WORKERS} parallel threads")
print("Estimated time: 30-45 minutes")
print()

sending_coords = {}

# Prepare list of cities to geocode
cities_to_geocode = [
    (row['Sending City'], row['Sending Country']) 
    for idx, row in sending_locations.iterrows()
]

# Parallel geocoding with progress bar
start_time = time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_to_city = {
        executor.submit(geocode_parallel, city, country): (city, country)
        for city, country in cities_to_geocode
    }
    
    for future in tqdm(as_completed(future_to_city), 
                       total=len(future_to_city), 
                       desc="Geocoding sending"):
        city, country = future_to_city[future]
        try:
            lat, lon = future.result()
            key = f"{city}|{country}"
            sending_coords[key] = {'lat': lat, 'lon': lon}
        except Exception as e:
            key = f"{city}|{country}"
            sending_coords[key] = {'lat': None, 'lon': None}

elapsed_time = time.time() - start_time

# Calculate statistics
geocoded_count = sum(1 for v in sending_coords.values() if v['lat'] is not None)
failed_count = len(sending_coords) - geocoded_count

print(f"\n{'=' * 70}")
print("SENDING LOCATIONS - GEOCODING RESULTS")
print(f"{'=' * 70}")
print(f"✓ Successfully geocoded: {geocoded_count:,} / {len(sending_locations):,} "
      f"({geocoded_count/len(sending_locations)*100:.1f}%)")
print(f"✗ Failed to geocode:     {failed_count:,} / {len(sending_locations):,} "
      f"({failed_count/len(sending_locations)*100:.1f}%)")
print(f"⏱️  Time taken:            {elapsed_time/60:.1f} minutes")
print(f"⚡ Speed:                 {len(sending_locations)/elapsed_time*60:.0f} locations/minute")
print(f"{'=' * 70}\n")

In [ ]:
# =============================================================================
# CELL 6: Geocode Receiving Locations
# =============================================================================

print("=" * 70)
print("GEOCODING RECEIVING LOCATIONS")
print("=" * 70)
print(f"Total unique receiving locations: {len(receiving_locations):,}")
print(f"Using {MAX_WORKERS} parallel threads")
print("Estimated time: 20-30 minutes")
print()

receiving_coords = {}

# Prepare list of cities to geocode
cities_to_geocode = [
    (row['Receiving City'], row['Receiving Country']) 
    for idx, row in receiving_locations.iterrows()
]

# Parallel geocoding with progress bar
start_time = time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_to_city = {
        executor.submit(geocode_parallel, city, country): (city, country)
        for city, country in cities_to_geocode
    }
    
    for future in tqdm(as_completed(future_to_city), 
                       total=len(future_to_city), 
                       desc="Geocoding receiving"):
        city, country = future_to_city[future]
        try:
            lat, lon = future.result()
            key = f"{city}|{country}"
            receiving_coords[key] = {'lat': lat, 'lon': lon}
        except Exception as e:
            key = f"{city}|{country}"
            receiving_coords[key] = {'lat': None, 'lon': None}

elapsed_time = time.time() - start_time

# Calculate statistics
geocoded_count = sum(1 for v in receiving_coords.values() if v['lat'] is not None)
failed_count = len(receiving_coords) - geocoded_count

print(f"\n{'=' * 70}")
print("RECEIVING LOCATIONS - GEOCODING RESULTS")
print(f"{'=' * 70}")
print(f"✓ Successfully geocoded: {geocoded_count:,} / {len(receiving_locations):,} "
      f"({geocoded_count/len(receiving_locations)*100:.1f}%)")
print(f"✗ Failed to geocode:     {failed_count:,} / {len(receiving_locations):,} "
      f"({failed_count/len(receiving_locations)*100:.1f}%)")
print(f"⏱️  Time taken:            {elapsed_time/60:.1f} minutes")
print(f"⚡ Speed:                 {len(receiving_locations)/elapsed_time*60:.0f} locations/minute")
print(f"{'=' * 70}\n")

In [ ]:
# =============================================================================
# CELL 7: Map Coordinates Back to Original Dataset
# =============================================================================

print("=" * 70)
print("MAPPING COORDINATES TO ORIGINAL DATASET")
print("=" * 70)

# Map sending coordinates
df_erasmus['Sending_Lat'] = df_erasmus.apply(
    lambda row: sending_coords.get(f"{row['Sending City']}|{row['Sending Country']}", {}).get('lat'),
    axis=1
)
df_erasmus['Sending_Lon'] = df_erasmus.apply(
    lambda row: sending_coords.get(f"{row['Sending City']}|{row['Sending Country']}", {}).get('lon'),
    axis=1
)

# Map receiving coordinates
df_erasmus['Receiving_Lat'] = df_erasmus.apply(
    lambda row: receiving_coords.get(f"{row['Receiving City']}|{row['Receiving Country']}", {}).get('lat'),
    axis=1
)
df_erasmus['Receiving_Lon'] = df_erasmus.apply(
    lambda row: receiving_coords.get(f"{row['Receiving City']}|{row['Receiving Country']}", {}).get('lon'),
    axis=1
)

print(f"✓ Coordinates mapped to dataset")
print(f"\nCoverage:")
print(f"  Sending:   {df_erasmus['Sending_Lat'].notna().sum():,} / {len(df_erasmus):,} "
      f"({df_erasmus['Sending_Lat'].notna().sum()/len(df_erasmus)*100:.1f}%)")
print(f"  Receiving: {df_erasmus['Receiving_Lat'].notna().sum():,} / {len(df_erasmus):,} "
      f"({df_erasmus['Receiving_Lat'].notna().sum()/len(df_erasmus)*100:.1f}%)")

# Save intermediate geocoded file
geocoded_output = f"{OUTPUT_DIR}/erasmus_geocoded.csv"
df_erasmus.to_csv(geocoded_output, index=False)
print(f"\n✓ Intermediate file saved: erasmus_geocoded.csv")

In [ ]:
# =============================================================================
# CELL 8: Load NUTS3 Boundaries
# =============================================================================

print("=" * 70)
print("LOADING NUTS3 BOUNDARIES")
print("=" * 70)

try:
    nuts3 = gpd.read_file(NUTS3_SHAPEFILE_PATH)
    print(f"✓ NUTS data loaded: {len(nuts3):,} regions")
    
    # Filter only NUTS3 level (level code = 3)
    nuts3 = nuts3[nuts3['LEVL_CODE'] == 3].copy()
    print(f"✓ NUTS3 regions (level 3): {len(nuts3):,}")
    print(f"✓ CRS: {nuts3.crs}")
    
    # Preview
    print("\nNUTS3 data preview:")
    display(nuts3[['NUTS_ID', 'NUTS_NAME', 'CNTR_CODE', 'LEVL_CODE']].head())
    
    nuts3_loaded = True
    
except FileNotFoundError:
    print(f"❌ ERROR: NUTS3 shapefile not found")
    print(f"   Expected: {NUTS3_SHAPEFILE_PATH}")
    print("\n📥 Download from:")
    print("   https://ec.europa.eu/eurostat/cache/GISCO/distribution/v2/nuts/shp/")
    print("   File: NUTS_RG_01M_2021_3035.shp.zip")
    print("\nUpdate NUTS3_SHAPEFILE_PATH in Cell 2 and re-run")
    nuts3_loaded = False

In [ ]:
# =============================================================================
# CELL 9: Spatial Join - Match Sending Locations to NUTS3
# =============================================================================

if nuts3_loaded:
    print("=" * 70)
    print("NUTS3 SPATIAL MATCHING - SENDING LOCATIONS")
    print("=" * 70)
    
    # Get valid geocoded sending locations
    sending_valid = df_erasmus[df_erasmus['Sending_Lat'].notna()].copy()
    print(f"Valid sending locations: {len(sending_valid):,}")
    
    if len(sending_valid) > 0:
        # Convert to GeoDataFrame
        print("Creating GeoDataFrame...")
        sending_gdf = gpd.GeoDataFrame(
            sending_valid,
            geometry=gpd.points_from_xy(sending_valid['Sending_Lon'], 
                                        sending_valid['Sending_Lat']),
            crs="EPSG:4326"
        )
        
        # Reproject to match NUTS CRS
        print(f"Reprojecting to {nuts3.crs}...")
        sending_gdf = sending_gdf.to_crs(nuts3.crs)
        
        # Spatial join
        print("Performing spatial join...")
        sending_matched = gpd.sjoin(
            sending_gdf,
            nuts3[['NUTS_ID', 'NUTS_NAME', 'CNTR_CODE', 'geometry']],
            how='left',
            predicate='within'
        )
        
        # Map back to original dataframe
        print("Mapping NUTS3 codes to dataset...")
        nuts_mapping = dict(zip(sending_matched.index, sending_matched['NUTS_ID']))
        nuts_name_mapping = dict(zip(sending_matched.index, sending_matched['NUTS_NAME']))
        
        df_erasmus['Sending_NUTS3'] = df_erasmus.index.map(nuts_mapping)
        df_erasmus['Sending_NUTS3_Name'] = df_erasmus.index.map(nuts_name_mapping)
        
        matched_count = df_erasmus['Sending_NUTS3'].notna().sum()
        print(f"\n{'=' * 70}")
        print(f"✓ Matched: {matched_count:,} / {len(df_erasmus):,} "
              f"({matched_count/len(df_erasmus)*100:.1f}%)")
        print(f"{'=' * 70}")
    else:
        print("⚠️  No valid sending locations to match")
else:
    print("⚠️  Skipping - NUTS3 data not loaded")

In [ ]:
# =============================================================================
# CELL 10: Spatial Join - Match Receiving Locations to NUTS3
# =============================================================================

if nuts3_loaded:
    print("=" * 70)
    print("NUTS3 SPATIAL MATCHING - RECEIVING LOCATIONS")
    print("=" * 70)
    
    # Get valid geocoded receiving locations
    receiving_valid = df_erasmus[df_erasmus['Receiving_Lat'].notna()].copy()
    print(f"Valid receiving locations: {len(receiving_valid):,}")
    
    if len(receiving_valid) > 0:
        # Convert to GeoDataFrame
        print("Creating GeoDataFrame...")
        receiving_gdf = gpd.GeoDataFrame(
            receiving_valid,
            geometry=gpd.points_from_xy(receiving_valid['Receiving_Lon'], 
                                         receiving_valid['Receiving_Lat']),
            crs="EPSG:4326"
        )
        
        # Reproject to match NUTS CRS
        print(f"Reprojecting to {nuts3.crs}...")
        receiving_gdf = receiving_gdf.to_crs(nuts3.crs)
        
        # Spatial join
        print("Performing spatial join...")
        receiving_matched = gpd.sjoin(
            receiving_gdf,
            nuts3[['NUTS_ID', 'NUTS_NAME', 'CNTR_CODE', 'geometry']],
            how='left',
            predicate='within'
        )
        
        # Map back to original dataframe
        print("Mapping NUTS3 codes to dataset...")
        nuts_mapping = dict(zip(receiving_matched.index, receiving_matched['NUTS_ID']))
        nuts_name_mapping = dict(zip(receiving_matched.index, receiving_matched['NUTS_NAME']))
        
        df_erasmus['Receiving_NUTS3'] = df_erasmus.index.map(nuts_mapping)
        df_erasmus['Receiving_NUTS3_Name'] = df_erasmus.index.map(nuts_name_mapping)
        
        matched_count = df_erasmus['Receiving_NUTS3'].notna().sum()
        print(f"\n{'=' * 70}")
        print(f"✓ Matched: {matched_count:,} / {len(df_erasmus):,} "
              f"({matched_count/len(df_erasmus)*100:.1f}%)")
        print(f"{'=' * 70}")
    else:
        print("⚠️  No valid receiving locations to match")
else:
    print("⚠️  Skipping - NUTS3 data not loaded")

In [ ]:
# =============================================================================
# CELL 11: Save Final Results and Summary
# =============================================================================

print("=" * 70)
print("SAVING FINAL RESULTS")
print("=" * 70)

# Save final dataset
output_path = f"{OUTPUT_DIR}/erasmus_with_nuts3.csv"
df_erasmus.to_csv(output_path, index=False)
print(f"✓ Final dataset saved: erasmus_with_nuts3.csv")
print(f"  Location: {output_path}")

# Print final summary
print("\n" + "=" * 70)
print("FINAL SUMMARY STATISTICS")
print("=" * 70)
print(f"Total records: {len(df_erasmus):,}")

print(f"\n📍 GEOCODING COVERAGE:")
print(f"  Sending:   {df_erasmus['Sending_Lat'].notna().sum():6,} / {len(df_erasmus):,} "
      f"({df_erasmus['Sending_Lat'].notna().sum()/len(df_erasmus)*100:5.1f}%)")
print(f"  Receiving: {df_erasmus['Receiving_Lat'].notna().sum():6,} / {len(df_erasmus):,} "
      f"({df_erasmus['Receiving_Lat'].notna().sum()/len(df_erasmus)*100:5.1f}%)")

if 'Sending_NUTS3' in df_erasmus.columns:
    print(f"\n🗺️  NUTS3 COVERAGE:")
    print(f"  Sending:   {df_erasmus['Sending_NUTS3'].notna().sum():6,} / {len(df_erasmus):,} "
          f"({df_erasmus['Sending_NUTS3'].notna().sum()/len(df_erasmus)*100:5.1f}%)")
    print(f"  Receiving: {df_erasmus['Receiving_NUTS3'].notna().sum():6,} / {len(df_erasmus):,} "
          f"({df_erasmus['Receiving_NUTS3'].notna().sum()/len(df_erasmus)*100:5.1f}%)")
    
    # Count complete records
    complete = df_erasmus[
        df_erasmus['Sending_NUTS3'].notna() & 
        df_erasmus['Receiving_NUTS3'].notna()
    ]
    print(f"\n✅ COMPLETE RECORDS (both NUTS3): {len(complete):,} / {len(df_erasmus):,} "
          f"({len(complete)/len(df_erasmus)*100:5.1f}%)")
    
    # Show unique NUTS3 regions covered
    unique_nuts3 = pd.concat([
        df_erasmus['Sending_NUTS3'].dropna(),
        df_erasmus['Receiving_NUTS3'].dropna()
    ]).nunique()
    print(f"\n📊 Unique NUTS3 regions covered: {unique_nuts3:,}")

print("=" * 70)
print("✅ GEOCODING AND NUTS3 MATCHING COMPLETE")
print("=" * 70)
print("\nNext steps:")
print("  1. Review failed geocoding cases (if any)")
print("  2. Proceed to LLM validation (notebook 02)")
print("  3. Apply manual corrections (notebook 03)")
print("  4. ETER institutional linkage (notebook 04)")
print("  5. Eurostat regional enrichment (notebook 05)")